In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
#cargar Data
customers_aditionals = pd.read_csv("bank-additional.csv")
pd.options.display.max_columns=30
customers_aditionals.head(3)

Ejecutamos una serie de comandos basicos para tener una idea general del data set antes de comenzar a trabajar

In [ ]:
customers_aditionals.info()

In [ ]:

customers_aditionals.describe()

In [ ]:
customers_aditionals['age'].value_counts()

comprobamos que las filas donde AGE no tienen datos sean de utilidad, ya qmue representan un 12% de nuestro data set y puede afectar gravemente los resultados

In [ ]:
customers_aditionals[customers_aditionals['age'].isna()].describe()

In [ ]:
customers_aditionals[["housing","loan"]]

In [ ]:
customers_aditionals[customers_aditionals['education'].isna()].head(5)

In [ ]:
customers_aditionals['education'].unique()

In [ ]:
customers_aditionals.isnull().sum()

In [ ]:
customers_aditionals["job"].unique()

In [ ]:
customers_aditionals[customers_aditionals['euribor3m'].isnull()].head(5)

In [ ]:
customers_aditionals['euribor3m'].unique()

In [ ]:
customers_aditionals["date"]

In [ ]:
customers_aditionals["cons.price.idx"].unique()

comenzamos la transformacion, quitando, rellenando o eliminando filas que tengan datos incompletos o no validos

rellenamos valores de trabajo y educacion con valores utilizables como "se abstiene" ya que las filas tienen valores validos

rellenamos datos como tasas de interes creditos con valores neutrales 

In [ ]:
customers_aditionals["job"] = customers_aditionals["job"].fillna("abstains")
customers_aditionals["default"] = customers_aditionals["default"].fillna(0)
customers_aditionals[["housing","loan"]] = customers_aditionals[["housing","loan"]].fillna(0)
customers_aditionals['education'] = customers_aditionals['education'].fillna("abstains")
customers_aditionals['marital'] = customers_aditionals['marital'].fillna("abstains")
customers_aditionals['euribor3m'] = customers_aditionals['euribor3m'].fillna(0)

#limpiamos y cambiamos de tipo la columna de indice de precios
customers_aditionals = customers_aditionals.dropna(subset=['cons.price.idx'])
customers_aditionals['cons.price.idx'] = customers_aditionals['cons.price.idx'].str.replace(',', '.', regex=False)
customers_aditionals['cons.price.idx'] = pd.to_numeric(customers_aditionals['cons.price.idx'], errors='coerce')
#cambiamos el tipo de dato de las columnas Cons.conf eribor3m nr.employed
for col in ['cons.conf.idx', 'euribor3m', 'nr.employed']:
    customers_aditionals[col] = customers_aditionals[col].astype(str).str.replace(',', '.', regex=False)
    customers_aditionals[col] = pd.to_numeric(customers_aditionals[col], errors='coerce')

customers_aditionals[['cons.price.idx','cons.conf.idx','euribor3m','nr.employed']].dtypes


In [ ]:
#Segmentar por edad, dejando los NaN como 'Sin dato' ---
bins = [0, 18, 30, 45, 60, 100]
labels = ['0-18', '19-30', '31-45', '46-60', '60+']

customers_aditionals['grupo_edad'] = pd.cut(customers_aditionals['age'], bins=bins, labels=labels)
customers_aditionals['grupo_edad'] = customers_aditionals['grupo_edad'].astype(str)
customers_aditionals['grupo_edad'] = customers_aditionals['grupo_edad'].replace('nan', 'Sin dato')

In [ ]:
#eliminamos datos faltantes ya que es una porcion pequeña del data set
customers_aditionals = customers_aditionals.dropna(subset=['date'])

#transformamos nuestra fecha en valores utilizables
meses = {
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04',
    'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
    'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
}

def convertir_fecha(fecha_str):
    dia, mes, año = fecha_str.split('-')
    return f"{dia}-{meses[mes.lower()]}-{año}"

customers_aditionals['date'] = customers_aditionals['date'].apply(convertir_fecha)

#Convertir a datetime
customers_aditionals['date'] = pd.to_datetime(customers_aditionals['date'], format='%d-%m-%Y')

In [ ]:
print(customers_aditionals['date'].dtype)
customers_aditionals['date'].head()


In [ ]:
customers_aditionals.isnull().sum()

In [ ]:
#Exportaremos el CSV limpio para trabajar en una sesion en blanco
customers_aditionals.to_csv(
    'Datos_Limpios.csv',
    index=False,          # no incluir el índice del DataFrame
    sep=',',              
    encoding='utf-8-sig', 
    header=True           # incluir (o no) los nombres de columna
)
